# Training

 Install packages

In [1]:
!uv pip install -q boto3==1.42.68 catboost==1.2.10 imbalanced-learn==0.14.1 lightgbm==4.6.0 matplotlib==3.10.8 mlflow==3.10.1 mlflow-skinny==3.10.1 mlflow-tracing==3.10.1 numpy==2.4.3 pandas==2.3.3 pyarrow==23.0.1 scikit-learn==1.8.0 seaborn==0.13.2 shap==0.51.0 xgboost==3.2.0

In [2]:
!uv pip freeze | grep -E "scikit-learn|xgboost|lightgbm|catboost|mlflow|boto3|pyarrow|pandas|numpy|matplotlib|seaborn|shap|imbalanced-learn"

boto3==1.42.68
catboost==1.2.10
imbalanced-learn==0.14.1
lightgbm==4.6.0
matplotlib==3.10.8
mlflow==3.10.1
mlflow-skinny==3.10.1
mlflow-tracing==3.10.1
numpy==2.4.3
pandas==2.3.3
pyarrow==23.0.1
scikit-learn==1.8.0
seaborn==0.13.2
shap==0.51.0
xgboost==3.2.0


Import packages

In [12]:
import io
import pickle
import warnings
from typing import Any, Dict, Tuple, Union

import boto3
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import shap
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

In [13]:
warnings.filterwarnings("ignore")

S3_ENDPOINT = "http://localstack:4566"
BUCKET = "data-lake"
PREFIX = "gold/credit_risk/features/ingestion_date=2026-03-14/"
TARGET = "serious_dlqin2yrs"
RANDOM_STATE: int = 42

MLFLOW_TRACKING_URI = "http://mlflow:5000"
EXPERIMENT_NAME = "credit_risk_training"

s3_client = boto3.client("s3")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(EXPERIMENT_NAME)

print(f"Experiment: {exp.name} (id={exp.experiment_id})")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Artifact store: {exp.artifact_location}")

Experiment: credit_risk_training (id=1)
Tracking URI: http://mlflow:5000
Artifact store: mlflow-artifacts:/1


## Evaluation Helper Functions

In [14]:
def ks_statistic(
    y_true: Union[np.ndarray, pd.Series], y_prob: Union[np.ndarray, pd.Series]
) -> float:
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)
    return float(np.max(true_positive_rate - false_positive_rate))


def gini(auc: float) -> float:
    return 2.0 * auc - 1.0


def evaluate(
    model: BaseEstimator,
    X: Union[np.ndarray, pd.DataFrame],
    y: Union[np.ndarray, pd.Series],
    split_name: str,
    log_to_mlflow: bool = True,
) -> Dict[str, float]:

    # Get probabilities for the positive class
    y_prob: np.ndarray = model.predict_proba(X)[:, 1]

    auc: float = float(roc_auc_score(y, y_prob))
    ks: float = ks_statistic(y, y_prob)
    pr_auc: float = float(average_precision_score(y, y_prob))
    g: float = gini(auc)

    metrics: Dict[str, float] = {
        f"{split_name}_auc_roc": round(auc, 4),
        f"{split_name}_ks": round(ks, 4),
        f"{split_name}_gini": round(g, 4),
        f"{split_name}_pr_auc": round(pr_auc, 4),
    }

    if log_to_mlflow:
        mlflow.log_metrics(metrics)

    print(
        f"  [{split_name}] AUC={auc:.4f} | KS={ks:.4f} | Gini={g:.4f} | PR-AUC={pr_auc:.4f}"
    )

    return metrics


def cv_score(
    model: BaseEstimator,
    X: Union[np.ndarray, pd.DataFrame],
    y: Union[np.ndarray, pd.Series],
    n_splits: int = 5,
) -> Tuple[float, float]:

    cv: StratifiedKFold = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # cross_val_score returns a numpy array of scores for each fold
    scores: np.ndarray = cross_val_score(
        model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1
    )

    mean_score: float = float(scores.mean())
    std_score: float = float(scores.std())

    print(f"  [CV-{n_splits}fold] AUC={mean_score:.4f} ± {std_score:.4f}")

    return mean_score, std_score

## Load Gold from S3

In [6]:
keys = [
    o["Key"]
    for o in s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", [])
    if o["Key"].endswith(".parquet")
]

df = pd.concat(
    [
        pd.read_parquet(
            io.BytesIO(s3_client.get_object(Bucket=BUCKET, Key=k)["Body"].read())
        )
        for k in keys
    ],
    ignore_index=True,
)

# drop partition column written by Spark — not a feature
df = df.drop(columns=["ingestion_date"], errors="ignore")

df.head()

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents,monthly_income_is_missing,number_of_dependents_is_missing,delinquency_score,debt_to_income_ratio,unsecured_to_total_lines_ratio,age_risk_bucket,has_any_delinquency
0,0,1.000000,49,0,2883.000000,0.0,9,0,2,0,1.0,0,0,0.0,NaN,0.777778,middle,0
1,1,1.000000,44,2,0.547762,6500.0,13,0,4,0,2.0,0,0,2.0,0.000084,0.692308,middle,1
2,0,0.001102,58,0,0.396173,3500.0,6,0,2,0,0.0,0,0,0.0,0.000113,0.666667,senior,0
3,0,0.018978,61,0,1129.000000,NaN,9,0,1,0,0.0,1,0,0.0,NaN,0.888889,senior,0
4,0,0.047006,40,0,0.183151,5388.0,12,0,0,0,0.0,0,0,0.0,0.000034,1.000000,middle,0


In [7]:
print(f"Shape: {df.shape}\n")
print(f"Columns: {df.columns.tolist()}\n")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}\n")
print(f"Default rate: {df[TARGET].mean()*100:.2f}%\n")

Shape: (149390, 18)

Columns: ['serious_dlqin2yrs', 'revolving_utilization_of_unsecured_lines', 'age', 'number_of_time30_59_days_past_due_not_worse', 'debt_ratio', 'monthly_income', 'number_of_open_credit_lines_and_loans', 'number_of_times90_days_late', 'number_real_estate_loans_or_lines', 'number_of_time60_89_days_past_due_not_worse', 'number_of_dependents', 'monthly_income_is_missing', 'number_of_dependents_is_missing', 'delinquency_score', 'debt_to_income_ratio', 'unsecured_to_total_lines_ratio', 'age_risk_bucket', 'has_any_delinquency']


Target distribution:
serious_dlqin2yrs
0    139381
1     10009
Name: count, dtype: int64

Default rate: 6.70%



## Stratified split

In [8]:
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

for name, ys in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:<7}: {len(ys):>7} rows | default rate: {ys.mean() * 100}")

train  :  104573 rows | default rate: 6.699626098514913
val    :   22408 rows | default rate: 6.698500535523028
test   :   22409 rows | default rate: 6.702664108170824


## Preprocessing Pipeline 

In [9]:
# Columns with nulls that need imputation
NUMERIC_WITH_NULLS = ["monthly_income", "number_of_dependents", "debt_to_income_ratio"]

# All numeric features (impute + scale)
NUMERIC_FEATURES = [
    "revolving_utilization_of_unsecured_lines",
    "age",
    "number_of_time30_59_days_past_due_not_worse",
    "debt_ratio",
    "monthly_income",
    "number_of_open_credit_lines_and_loans",
    "number_of_times90_days_late",
    "number_real_estate_loans_or_lines",
    "number_of_time60_89_days_past_due_not_worse",
    "number_of_dependents",
    "monthly_income_is_missing",
    "number_of_dependents_is_missing",
    "delinquency_score",
    "debt_to_income_ratio",
    "unsecured_to_total_lines_ratio",
    "has_any_delinquency",
]

# Categorical features
CATEGORICAL_FEATURES = ["age_risk_bucket"]

# Age bucket order matters for ordinal encoding
AGE_BUCKET_ORDER = [["young", "middle", "senior", "elderly"]]

In [10]:
numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        (
            "encoder",
            OrdinalEncoder(
                categories=AGE_BUCKET_ORDER,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        )
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
x_val_proc = preprocessor.transform(X_val)
x_test_proc = preprocessor.transform(X_test)

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Preprocessor fitted on {len(X_train)} rows.")
print(f"Features after preprocessing: {X_train_proc.shape[1]}")
print(f"\nImputed medians (train):")

medians = preprocessor.named_transformers_["numeric"]["imputer"].statistics_
for fname, median in zip(NUMERIC_FEATURES, medians):
    if fname in NUMERIC_WITH_NULLS:
        print(f"  {fname}: {median:.4f}")

Preprocessor fitted on 104573 rows.
Features after preprocessing: 17

Imputed medians (train):
  monthly_income: 5400.0000
  number_of_dependents: 0.0000
  debt_to_income_ratio: 0.0000


## Model Definition

In [18]:
# scale_pos_weight = neg/pos = 13.9 (from EDA)
SPW = 13.9

MODELS = {
    "logistic_regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
    ),
    "xgboost": XGBClassifier(
        scale_pos_weight=SPW,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        early_stopping_rounds=20,
        random_state=RANDOM_STATE,
    ),
    "lightgbm": LGBMClassifier(
        scale_pos_weight=SPW,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbpsity=-1,
    ),
    "catboost": CatBoostClassifier(
        scale_pos_weight=SPW,
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=RANDOM_STATE,
        verbose=0,
    ),
}

print("Models defined:")
for name in MODELS:
    print(f" {name}")
print(f"\nscale_pos_weight = {SPW} (neg/pos ratio from EDA")

Models defined:
 logistic_regression
 xgboost
 lightgbm
 catboost

scale_pos_weight = 13.9 (neg/pos ratio from EDA


## Training